# Μάθημα 12 - Μείωση Ιστορικού Συνομιλίας με Agent Scratchpad

Αυτό το σημειωματάριο δείχνει πώς να διαχειρίζεστε το πλαίσιο σε μακριές συνομιλίες χρησιμοποιώντας το Microsoft Agent Framework. Καθώς οι συνομιλίες μεγαλώνουν, ο αριθμός των token αυξάνεται — τελικά ξεπερνώντας το παράθυρο πλαισίου του μοντέλου. Αντιμετωπίζουμε αυτό με ένα **πρότυπο περίληψης πλαισίου** και ένα **agent scratchpad** για επίμονη μνήμη.

## Τι θα Μάθετε:
1. **Γιατί η Διαχείριση Πλαισίου Έχει Σημασία**: Κατανόηση των ορίων token και των παραθύρων πλαισίου
2. **Πράκτορες με Επίγνωση Πλαισίου**: Δημιουργία πρακτόρων που διαχειρίζονται το δικό τους πλαίσιο συνομιλίας
3. **Πρότυπο Περίληψης Πλαισίου**: Χρήση εργαλείων για τη συμπύκνωση του ιστορικού συνομιλίας
4. **Agent Scratchpad**: Επίμονη μνήμη που επιβιώνει της μείωσης πλαισίου

## Προαπαιτούμενα:
- Ρύθμιση Azure OpenAI με διαμορφωμένες μεταβλητές περιβάλλοντος
- Κατανόηση βασικών εννοιών πρακτόρων από προηγούμενα μαθήματα


## Ρύθμιση


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Γιατί η Διαχείριση του Πλαισίου Έχει Σημασία

Κάθε LLM έχει ένα πεπερασμένο **παράθυρο πλαισίου** — τον μέγιστο αριθμό tokens που μπορεί να επεξεργαστεί σε ένα μόνο αίτημα. Καθώς εξελίσσεται μια συνομιλία πολλαπλών γύρων:

- **Ο αριθμός των tokens αυξάνεται γραμμικά** με κάθε μήνυμα χρήστη και απάντηση βοηθού.
- **Τα tokens του prompt κυριαρχούν στο κόστος** επειδή όλη η ιστορία επαναστέλνεται κάθε φορά.
- Τελικά η συνομιλία **ξεπερνά το παράθυρο πλαισίου** και το μοντέλο είτε περικόπτει είτε προκαλεί σφάλμα.

### Στρατηγικές για τη Διαχείριση του Πλαισίου

| Στρατηγική | Πώς Λειτουργεί | Αντισταθμιστικό Όφελος |
|---|---|---|
| **Περικοπή** | Απορρίπτει τα παλαιότερα μηνύματα | Χάνει πρώιμο πλαίσιο |
| **Περίληψη** | Συνοψίζει τα παλαιότερα μηνύματα σε μια περίληψη | Χάνονται κάποιες λεπτομέρειες, αλλά διατηρούνται τα βασικά σημεία |
| **Scratchpad / Εξωτερική Μνήμη** | Αποθηκεύει βασικά στοιχεία έξω από τη συνομιλία | Απαιτεί κλήσεις εργαλείων, αλλά διατηρείται ανεξαρτήτως περικοπής |

Σε αυτό το σημειωματάριο συνδυάζουμε την **περίληψη** με ένα **εργαλείο scratchpad** ώστε ο πράκτορας να μπορεί να διατηρεί συνέχεια ακόμα και όταν η ιστορία της συνομιλίας συμπυκνώνεται.


## Δημιουργία ενός Πράκτορα με Επίγνωση Συμφραζομένων


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Προσομοίωση Μιας Μακράς Συζήτησης

Ας περάσουμε μέσα από μια συνομιλία πολλαπλών γύρων για να δούμε πώς συσσωρεύεται το πλαίσιο. Ο πράκτορας θα πρέπει να διατηρεί βασικές λεπτομέρειες (προτιμήσεις, προϋπολογισμό, ημερομηνίες ταξιδιού) κατά τη διάρκεια των γύρων και να επιδεικνύει συνέχεια.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Παρατηρήστε πώς ο πράκτορας διατηρεί το πλαίσιο από προηγούμενες γύρους — γνωρίζει για την Ιαπωνία, το σούσι, τα ναούς, τη φωτογραφία, τον προϋπολογισμό των $3000, το ταξίδι μόνος, και το χρονικό πλαίσιο του Απριλίου. Σε μια σύντομη συνομιλία αυτό λειτουργεί καλά, αλλά καθώς η συνομιλία μεγαλώνει, το πλήρες ιστορικό γίνεται δαπανηρό να ξανασταλεί.

Ας συνεχίσουμε τη συνομιλία με περισσότερους γύρους για να δούμε τη συσσώρευση του πλαισίου:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Πρότυπο Περίληψης Πλαισίου

Καθώς η συνομιλία μεγαλώνει, μπορούμε να χρησιμοποιήσουμε ένα **εργαλείο περιλήψεων** για να συμπτύξουμε το συσσωρευμένο πλαίσιο σε μια συμπαγή μορφή. Ο παράγοντας καλεί αυτό το εργαλείο για να καταγράψει βασικές προτιμήσεις ώστε ακόμα κι αν παλαιότερα μηνύματα διαγραφούν, οι ουσιώδεις πληροφορίες να διατηρούνται.

Αυτό το πρότυπο είναι ο δομικός λίθος για πιο εξεζητημένη μείωση του ιστορικού:
1. Ο παράγοντας εντοπίζει βασικά γεγονότα από τη συνομιλία
2. Καλεί το εργαλείο περίληψης για να τα διατηρήσει
3. Παλαιότερα μηνύματα μπορούν να αφαιρεθούν με ασφάλεια επειδή η περίληψη καταγράφει τα σημαντικά

Παρακάτω ορίζουμε ένα εργαλείο `summarize_preferences` που ο παράγοντας μπορεί να καλεί για να καταγράψει μια συμπαγή περίληψη αυτού που έχει μάθει.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Περίληψη

Σε αυτό το μάθημα μάθατε πώς να διαχειρίζεστε το πλαίσιο σε μακροχρόνιες συνομιλίες πρακτόρων χρησιμοποιώντας το Microsoft Agent Framework:

### Κύριες Έννοιες
- **Τα παράθυρα πλαισίου είναι πεπερασμένα** — κάθε τοκέν στην ιστορία της συνομιλίας κοστίζει χρήματα και μετράει προς το όριο.
- **Εργαλεία σύνοψης** επιτρέπουν στον πράκτορα να συμπυκνώνει το συσσωρευμένο πλαίσιο σε συμπαγείς συνοψίσεις, μειώνοντας τη χρήση τοκέν ενώ διατηρεί τις ουσιώδεις πληροφορίες.
- **Σημειωματάρια πρακτόρων** παρέχουν επίμονη εξωτερική μνήμη που επιβιώνει από οποιαδήποτε μείωση της συνομιλίας.

### Τι Δημιουργήσατε
- Έναν **πράκτορα που γνωρίζει το πλαίσιο** που διατηρεί τη συνέχεια σε συνομιλίες πολλαπλών γύρων
- Ένα **εργαλείο σύνοψης** (`summarize_preferences`) που καταγράφει βασικές λεπτομέρειες χρήστη σε συμπαγή μορφή
- Μια **συνομιλία πολλαπλών γύρων** που δείχνει διατήρηση πλαισίου και χειρισμό αλλαγών

### Πραγματικές Εφαρμογές
- **Bots Εξυπηρέτησης Πελατών**: Θυμούνται προτιμήσεις σε μακρές συνεδρίες υποστήριξης
- **Προσωπικοί Βοηθοί**: Παρακολουθούν τρέχοντα έργα χωρίς να επεξηγούν ξανά το πλαίσιο
- **Εκπαιδευτικοί Δάσκαλοι**: Διατηρούν την πρόοδο του μαθητή σε πολλές αλληλεπιδράσεις

### Επόμενα Βήματα
- Υλοποιήστε ένα πλήρες εργαλείο σημειωματαρίου με ανθεκτικότητα σε αρχεία
- Προσθέστε αυτόματη περικοπή ιστορικού μετά τη σύνοψη
- Συνδυάστε με βάσεις δεδομένων διανυσμάτων για αναζήτηση σημασιολογικής μνήμης
- Δημιουργήστε πράκτορες που μπορούν να συνεχίσουν συνομιλίες μέρες αργότερα με πλήρες πλαίσιο


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
